# In Class Activity April 14th, 2026

In [2]:
%pip install optuna

Note: you may need to restart the kernel to use updated packages.


### Importing libraries, preparing data, initial EDA

In [3]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


c:\Users\tyler\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
# importing data
adult = pd.read_csv('../datasets/adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [5]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





Workclass is very unbalanced, private dominates. This means I will need to deal with the class imbalance in the model construction. There are 52 duplicates in the dataset so I will need to clean and get rid of duplicate rows. I will change all "?" to NaN values and figure out to either use an average for that value or remove them.

### Data Preprocessing (minimal) and Baseline Model

In [6]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# encode categoricals as integer codes for tree models
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')
for col in categorical_cols:
    adult[col] = adult[col].cat.codes

adult.head(20)

C:\Users\tyler\AppData\Local\Temp\ipykernel_34496\1384556593.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = adult.select_dtypes(include='object').columns


,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,3,226802,7,4,6,3,2,1,0,0,40,38,0
1,38,3,89814,9,2,4,0,4,1,0,0,50,38,0
2,28,1,336951,12,2,10,0,4,1,0,0,40,38,1
3,44,3,160323,10,2,6,0,2,1,7688,0,40,38,1
4,18,-1,103497,10,4,-1,3,4,0,0,0,30,38,0
5,34,3,198693,6,4,7,1,4,1,0,0,30,38,0
6,29,-1,227026,9,4,-1,4,2,1,0,0,40,38,0
7,63,5,104626,15,2,9,0,4,1,3103,0,32,38,1
8,24,3,369667,10,4,7,4,4,0,0,0,40,38,0
9,55,3,104996,4,2,2,0,4,1,0,0,10,38,0


In [7]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [8]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70642626 0.71859064 0.7157021  0.71897555 0.71461397]
Mean F1 score: 0.7148617033494202


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The baseline model did an ok job and gave us a starting F1 score for the default hyperparameters and confirms that the model is learning some signal, but there is room for improvement.

To improve the baseline, I will:
- tune important XGBoost hyperparameters such as `max_depth`, `learning_rate`, `n_estimators`, and `scale_pos_weight`
- compare the models with more features and see how they effect F1
- evaluate performance with more than one metric (F1, precision, recall) to avoid overfitting to a single score

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [13]:
# Explore different XGBoost hyperparameters with stratified cross-validation
experiments = [
    {
        'name': 'default',
        'model': XGBClassifier(random_state=42)
    },
    {
        'name': 'shallow_tree',
        'model': XGBClassifier(max_depth=3, learning_rate=0.1, n_estimators=100, random_state=42)
    },
    {
        'name': 'deeper_tree',
        'model': XGBClassifier(max_depth=7, learning_rate=0.1, n_estimators=100, random_state=42)
    },
    {
        'name': 'lower_learning_rate',
        'model': XGBClassifier(max_depth=5, learning_rate=0.01, n_estimators=200, random_state=42)
    },
    {
        'name': 'balanced_class_weights',
        'model': XGBClassifier(
            scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
            max_depth=5,
            learning_rate=0.1,
            n_estimators=150,
            random_state=42
        )
    },
    {
        'name': 'regularized',
        'model': XGBClassifier(
            max_depth=5,
            learning_rate=0.1,
            n_estimators=150,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        )
    }
]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for exp in experiments:
    scores = cross_val_score(exp['model'], X, y, cv=skf, scoring='f1', n_jobs=-1)
    results.append({
        'model': exp['name'],
        'mean_f1': scores.mean(),
        'std_f1': scores.std(),
        'scores': scores
    })

results_df = pd.DataFrame(results).sort_values('mean_f1', ascending=False).reset_index(drop=True)
print(results_df[['model', 'mean_f1', 'std_f1']])

best_model_name = results_df.loc[0, 'model']
print(f"\nBest model: {best_model_name} with mean F1 = {results_df.loc[0, 'mean_f1']:.4f}")

                    model   mean_f1    std_f1
0                 default  0.714862  0.004533
1             regularized  0.713808  0.007662
2  balanced_class_weights  0.713085  0.005250
3             deeper_tree  0.712952  0.005947
4            shallow_tree  0.676949  0.003899
5     lower_learning_rate  0.635393  0.009138

Best model: default with mean F1 = 0.7149


Surprisingly, the default model performed the best still. This is interesting and could be due to poor tuning parameters and the model correcting itself too much causing poorer performance. The models may be overfitting so I will need to explore more options before I'd be confident submitting my model

### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [14]:
# preferred model from the earlier exploration is the default XGBoost classifier
base_model = XGBClassifier(random_state=42)

# parameter grid for 4-5 important XGBoost hyperparameters
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 150, 200],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9]
}

grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='f1',
    cv=skf,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print('Best GridSearchCV parameters:')
print(grid_search.best_params_)
print(f'Best cross-validated F1: {grid_search.best_score_:.4f}')

best_grid_model = grid_search.best_estimator_

# Evaluate the final tuned model on the held-out test set
best_grid_model.fit(X_train, y_train)
y_pred_grid = best_grid_model.predict(X_test)

print('\nTest set performance for GridSearchCV-tuned model:')
print(classification_report(y_test, y_pred_grid))
print(f'Test F1 score: {f1_score(y_test, y_pred_grid):.4f}')

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best GridSearchCV parameters:
{'colsample_bytree': 0.7, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 150, 'subsample': 0.9}
Best cross-validated F1: 0.7145

Test set performance for GridSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.90      0.95      0.92      7431
           1       0.79      0.65      0.71      2338

    accuracy                           0.88      9769
   macro avg       0.84      0.80      0.82      9769
weighted avg       0.87      0.88      0.87      9769

Test F1 score: 0.7145


### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [16]:
# tuning xgboost classifier with RandomizedSearchCV using the preferred default model
param_dist = {
    'max_depth': np.arange(3, 9),
    'learning_rate': np.linspace(0.01, 0.2, num=10),
    'n_estimators': [100, 150, 200],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

xgb_random = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=30,
    cv=skf,
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_random.fit(X_train, y_train)
print('Best parameters from RandomizedSearchCV:')
print(xgb_random.best_params_)
print(f'Best cross-validated F1: {xgb_random.best_score_:.4f}')

# build the final tuned model using the best parameters and evaluate on the test set
xgb_random_best = XGBClassifier(random_state=42,
                                **xgb_random.best_params_)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)

print('\nTest set performance for RandomizedSearchCV-tuned model:')
print(classification_report(y_test, y_pred_random))
print(f'Test F1 score: {f1_score(y_test, y_pred_random):.4f}')

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters from RandomizedSearchCV:
{'subsample': 0.8, 'n_estimators': 200, 'max_depth': np.int64(6), 'learning_rate': np.float64(0.11555555555555555), 'colsample_bytree': 0.8}
Best cross-validated F1: 0.7125

Test set performance for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      7431
           1       0.79      0.65      0.71      2338

    accuracy                           0.87      9769
   macro avg       0.84      0.80      0.82      9769
weighted avg       0.87      0.87      0.87      9769

Test F1 score: 0.7142


### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [17]:
# tuning with Optuna for the preferred XGBoost model and the same 4-5 hyperparameters used above

def objective(trial):
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2)
    n_estimators = trial.suggest_int('n_estimators', 100, 200)
    subsample = trial.suggest_float('subsample', 0.7, 0.9)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.7, 0.9)

    xgb_optuna = XGBClassifier(
        random_state=42,
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree
    )

    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1', n_jobs=-1)
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print('Best parameters from Optuna:')
print(study.best_params)
print(f'Best cross-validated F1 from Optuna: {study.best_value:.4f}')

# train the final tuned model with the best parameters and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, **study.best_params)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)

print('\nTest set performance for Optuna-tuned model:')
print(classification_report(y_test, y_pred_optuna))
print(f'Test F1 score: {f1_score(y_test, y_pred_optuna):.4f}')


[I 2026-04-15 15:40:23,267] A new study created in memory with name: no-name-47224742-f37b-4c80-bea6-48c852314dcf


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-15 15:40:24,258] Trial 0 finished with value: 0.7027556716024723 and parameters: {'max_depth': 4, 'learning_rate': 0.07486278735927222, 'n_estimators': 157, 'subsample': 0.8715583826759632, 'colsample_bytree': 0.8876469906206859}. Best is trial 0 with value: 0.7027556716024723.
[I 2026-04-15 15:40:25,507] Trial 1 finished with value: 0.7086028194412091 and parameters: {'max_depth': 9, 'learning_rate': 0.12075553088750018, 'n_estimators': 132, 'subsample': 0.7513097378462287, 'colsample_bytree': 0.8145150407463313}. Best is trial 1 with value: 0.7086028194412091.
[I 2026-04-15 15:40:26,600] Trial 2 finished with value: 0.6450542399974596 and parameters: {'max_depth': 4, 'learning_rate': 0.026253273695167265, 'n_estimators': 157, 'subsample': 0.7907533889810373, 'colsample_bytree': 0.8453358974097149}. Best is trial 1 with value: 0.7086028194412091.
[I 2026-04-15 15:40:27,639] Trial 3 finished with value: 0.7092842929761831 and parameters: {'max_depth': 10, 'learning_rate': 0.

### Tuning results

The tuning process showed that different search methods can behave differently on this dataset. GridSearchCV gave a reliable, systematic search over a smaller grid and produced strong results, while RandomizedSearchCV explored a wider range of hyperparameter combinations more quickly. Optuna was the most flexible and efficient at finding high-performing settings because it adapted the search based on previous trials.

Based on model performance, the best results came from the method that yielded the highest cross-validated F1 and the best test-set classification report. I prefer Optuna for this task because it can focus the search on promising hyperparameter regions while still tuning the same 4-5 parameters, but GridSearchCV is useful when I want a clear, exhaustive comparison over a defined set of values.